In [6]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from pathlib import Path
import joblib
import xgboost as xgb
from data.preprocessing.pipeline import Pipeline

pipeline = Pipeline(download=False)
train, test = pipeline.get_processed_data()

ID_COL = "Pubchem_ID"
TARGET = "taste_cluster"

feature_cols = [
    c for c in train.columns
    if c not in [ID_COL, TARGET, "SMILES"]
    and train[c].dtype != "object"
]

print(f"Используем {len(feature_cols)} фичей")

X = train[feature_cols].values
y = train[TARGET].values
X_test = test[feature_cols].values

ids_test = test[ID_COL].values

SEED = 42
N_FOLDS = 5
N_CLASSES = len(np.unique(y))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

xgb_params = {
    "n_estimators": 600,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbosity": 0,
    "eval_metric": "auc"
}

oof_probs = np.zeros((len(train), N_CLASSES))
test_probs = np.zeros((len(test), N_CLASSES))

os.makedirs("models_out", exist_ok=True)

for class_idx in range(N_CLASSES):
    print(f"\n>>>> Обучаем XGB для класса {class_idx}")

    y_bin = (y == class_idx).astype(int)
    fold_test = np.zeros((len(test), N_FOLDS))

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        print(f"   Фолд {fold+1}/{N_FOLDS}")

        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y_bin[tr], y_bin[val]

        scaler = StandardScaler().fit(X_tr)
        X_tr_s = scaler.transform(X_tr)
        X_val_s = scaler.transform(X_val)
        X_test_s = scaler.transform(X_test)

        model = xgb.XGBClassifier(
            objective="binary:logistic",
            n_estimators=600,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            verbosity=0,
            eval_metric="auc",
            early_stopping_rounds=40,
        )
        
        model.fit(
            X_tr_s, y_tr,
            eval_set=[(X_val_s, y_val)],
        )

        oof_probs[val, class_idx] = model.predict_proba(X_val_s)[:, 1]

        fold_test[:, fold] = model.predict_proba(X_test_s)[:, 1]

    test_probs[:, class_idx] = fold_test.mean(axis=1)

print("\n>>> Обучаем мета-классификатор (LogReg)")

meta = LogisticRegression(max_iter=2000, multi_class="multinomial")
meta.fit(oof_probs, y)

oof_pred = meta.predict(oof_probs)
macro_f1 = f1_score(y, oof_pred, average="macro")
print(f"\nOOF Macro F1 = {macro_f1:.5f}")

final_pred = meta.predict(test_probs)

solution = pd.DataFrame({
    ID_COL: ids_test,
    TARGET: final_pred
})

solution.to_csv("solution.csv", index=False)
print("Готово! Сохранён файл solution.csv")

Extracting raw features:   0%|                                                                 | 0/2064 [00:00<?, ?it/s][17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGenerator
[17:02:55] DEPRECATION WARNING: please use MorganGener

Используем 2099 фичей

>>>> Обучаем XGB для класса 0
   Фолд 1/5
[0]	validation_0-auc:0.73176
[1]	validation_0-auc:0.74020
[2]	validation_0-auc:0.74703
[3]	validation_0-auc:0.74588
[4]	validation_0-auc:0.75169
[5]	validation_0-auc:0.75209
[6]	validation_0-auc:0.75738
[7]	validation_0-auc:0.76320
[8]	validation_0-auc:0.76160
[9]	validation_0-auc:0.76089
[10]	validation_0-auc:0.76088
[11]	validation_0-auc:0.76255
[12]	validation_0-auc:0.76206
[13]	validation_0-auc:0.76383
[14]	validation_0-auc:0.76567
[15]	validation_0-auc:0.76672
[16]	validation_0-auc:0.76595
[17]	validation_0-auc:0.76770
[18]	validation_0-auc:0.76752
[19]	validation_0-auc:0.76595
[20]	validation_0-auc:0.76699
[21]	validation_0-auc:0.76672
[22]	validation_0-auc:0.76899
[23]	validation_0-auc:0.77047
[24]	validation_0-auc:0.77034
[25]	validation_0-auc:0.77214
[26]	validation_0-auc:0.77179
[27]	validation_0-auc:0.77295
[28]	validation_0-auc:0.77305
[29]	validation_0-auc:0.77351
[30]	validation_0-auc:0.77240
[31]	validation